# Chapter 14
## Model Neurons of Bifurcation Type 2
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter14.ipynb)

## About this chapter

Type-2 excitability (onset via a Hopf bifurcation, non-zero onset
frequency) is studied in reduced two-dimensional HH and Erisir models
($m=m_\infty(v)$, and $h$ replaced by a constant-sum approximation of
$h+n$). Fixed points are tracked across $I_{ext}$, classified by their
Jacobian eigenvalues, and the coexisting attracting/repelling limit cycles
near a subcritical Hopf point are traced out directly.

See [`README.md`](chapter14.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact
from mnd.core import draw_arrow

## Reduced HH gating (shared by the `HH_REDUCED_*` examples below)

Classical-HH gating with $m=m_\infty(v)$, $h=0.83-n$.

In [ ]:
def hh_reduced_alpha_m(v):
    # removable singularity at v=-45 (0/0) -> L'Hopital limit is 1
    with np.errstate(divide='ignore', invalid='ignore'):
        out = (v + 45) / 10.0 / (1 - exp(-(v + 45) / 10))
    return np.where(np.abs(v + 45) > 1e-8, out, 1.0)


def hh_reduced_alpha_m_p(v):
    num, den = (v + 45) / 10, 1 - exp(-(v + 45) / 10)
    num_p, den_p = 1 / 10, exp(-(v + 45) / 10) / 10
    return (den * num_p - num * den_p) / den ** 2


def hh_reduced_beta_m(v):
    return 4 * exp(-(v + 70) / 18)


def hh_reduced_beta_m_p(v):
    return -(4 / 18) * exp(-(v + 70) / 18)


def hh_reduced_alpha_n(v):
    return 0.01 * (-60.0 - v) / (exp((-60 - v) / 10) - 1)


def hh_reduced_alpha_n_p(v):
    num, den = 0.01 * (-60.0 - v), exp((-60 - v) / 10) - 1
    num_p, den_p = -0.01, -(den + 1) * 0.1
    return (den * num_p - num * den_p) / den ** 2


def hh_reduced_beta_n(v):
    return 0.125 * exp(-(v + 70) / 80)


def hh_reduced_beta_n_p(v):
    return -hh_reduced_beta_n(v) / 80


def hh_reduced_m_inf(v):
    return hh_reduced_alpha_m(v) / (hh_reduced_alpha_m(v) + hh_reduced_beta_m(v))


def hh_reduced_m_inf_p(v):
    num = (hh_reduced_alpha_m(v) + hh_reduced_beta_m(v)) * hh_reduced_alpha_m_p(v)
    num = num - hh_reduced_alpha_m(v) * (hh_reduced_alpha_m_p(v) + hh_reduced_beta_m_p(v))
    return num / (hh_reduced_alpha_m(v) + hh_reduced_beta_m(v)) ** 2


def hh_reduced_n_inf(v):
    return hh_reduced_alpha_n(v) / (hh_reduced_alpha_n(v) + hh_reduced_beta_n(v))


def hh_reduced_f(v, g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0):
    """dv/dt at i_ext=0 of the reduced (m=m_inf(v), h=0.83-n_inf(v)) HH model"""
    return (g_na * hh_reduced_m_inf(v) ** 3 * (0.83 - hh_reduced_n_inf(v)) * (v_na - v)
            + g_k * hh_reduced_n_inf(v) ** 4 * (v_k - v) + g_l * (v_l - v))


def hh_reduced_find_fixed_point(i_ext, g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0):
    kw = dict(g_na=g_na, g_k=g_k, g_l=g_l, v_na=v_na, v_k=v_k, v_l=v_l)
    v_left, v_right = -100.0, 50.0
    while v_right - v_left > 1e-10:
        v_c = (v_left + v_right) / 2
        if (hh_reduced_f(v_c, **kw) + i_ext) * (hh_reduced_f(v_left, **kw) + i_ext) > 0:
            v_left = v_c
        else:
            v_right = v_c
    return (v_left + v_right) / 2


def hh_reduced_jacobian(v_c, n_c, c=1.0, g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0):
    j00 = (g_na * 3 * hh_reduced_m_inf(v_c) ** 2 * hh_reduced_m_inf_p(v_c) * (0.83 - n_c) * (v_na - v_c)
           - g_na * hh_reduced_m_inf(v_c) ** 3 * (0.83 - n_c) - g_k * n_c ** 4 - g_l)
    j01 = -g_na * hh_reduced_m_inf(v_c) ** 3 * (v_na - v_c) + 4 * g_k * n_c ** 3 * (v_k - v_c)
    j10 = hh_reduced_alpha_n_p(v_c) * (1 - n_c) - hh_reduced_beta_n_p(v_c) * n_c
    j11 = -hh_reduced_alpha_n(v_c) - hh_reduced_beta_n(v_c)
    return np.array([[j00, j01], [j10, j11]]) / c


def hh_reduced_simulate(v0, n0, i_ext, t_final, dt=0.01, direction=1,
                         c=1.0, g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    v[0], n[0] = v0, n0
    for k in range(m_steps):
        m_k = hh_reduced_m_inf(v[k])
        h_k = 0.83 - n[k]
        v_inc = (g_na * m_k ** 3 * h_k * (v_na - v[k]) + g_k * n[k] ** 4 * (v_k - v[k])
                 + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = hh_reduced_alpha_n(v[k]) * (1 - n[k]) - hh_reduced_beta_n(v[k]) * n[k]
        v_tmp = v[k] + direction * dt05 * v_inc
        n_tmp = n[k] + direction * dt05 * n_inc
        m_tmp = hh_reduced_m_inf(v_tmp)
        h_tmp = 0.83 - n_tmp
        v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        n_inc = hh_reduced_alpha_n(v_tmp) * (1 - n_tmp) - hh_reduced_beta_n(v_tmp) * n_tmp
        v[k + 1] = v[k] + direction * dt * v_inc
        n[k + 1] = n[k] + direction * dt * n_inc
    return v, n

## Erisir 2D Fixed-Point Bifurcation Diagram

In [ ]:
def simulate_erisir_2d_fp(c=1.0, g_k=224.0, g_na=112.0, g_l=0.5,
                           v_k=-90.0, v_na=60.0, v_l=-70.0, i_ext_vec=None):
    if i_ext_vec is None:
        i_ext_vec = np.arange(1001) / 1000 * 7

    def alpha_m(v):
        return 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)

    def alpha_m_p(v):
        num, den = 40 * (75.5 - v), exp((75.5 - v) / 13.5) - 1
        num_p, den_p = -40.0, -exp((75.5 - v) / 13.5) / 13.5
        return (den * num_p - num * den_p) / den ** 2

    def beta_m(v):
        return 1.2262 / exp(v / 42.248)

    def beta_m_p(v):
        return -beta_m(v) / 42.248

    def alpha_n(v):
        return (95 - v) / (exp((95 - v) / 11.8) - 1)

    def alpha_n_p(v):
        num, den = 95 - v, exp((95 - v) / 11.8) - 1
        num_p, den_p = -1.0, -(den + 1) / 11.8
        return (den * num_p - num * den_p) / den ** 2

    def beta_n(v):
        return 0.025 / exp(v / 22.222)

    def beta_n_p(v):
        return -beta_n(v) / 22.222

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def m_inf_p(v):
        num = (alpha_m(v) + beta_m(v)) * alpha_m_p(v)
        num = num - alpha_m(v) * (alpha_m_p(v) + beta_m_p(v))
        return num / (alpha_m(v) + beta_m(v)) ** 2

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def f(v, i_ext):
        """dv/dt of the reduced (m=m_inf(v), h=0.36-n_inf(v)) Erisir model"""
        return (g_na * m_inf(v) ** 3 * (0.36 - n_inf(v)) * (v_na - v)
                + g_k * n_inf(v) ** 2 * (v_k - v) + g_l * (v_l - v) + i_ext)

    def find_fixed_points(i_ext):
        v_min = min(v_k, v_l + i_ext / g_l)
        v_max = max(v_na, v_l + i_ext / g_l)
        n_v = 1000
        v_vec = v_min + np.arange(n_v + 1) / n_v * (v_max - v_min)
        f_vec = f(v_vec, i_ext)
        roots = []
        for i in np.where(f_vec[:-1] * f_vec[1:] <= 0)[0]:
            a, b_ = v_vec[i], v_vec[i + 1]
            while b_ - a > 1e-12:
                m = (a + b_) / 2
                if f(m, i_ext) * f(a, i_ext) <= 0:
                    b_ = m
                else:
                    a = m
            roots.append((a + b_) / 2)
        return roots

    def jacobian(v, n):
        j00 = (g_na * 3 * m_inf(v) ** 2 * m_inf_p(v) * (0.36 - n) * (v_na - v)
               - g_na * m_inf(v) ** 3 * (0.36 - n) - g_k * n ** 2 - g_l)
        j01 = -g_na * m_inf(v) ** 3 * (v_na - v) + g_k * 2 * n * (v_k - v)
        j10 = alpha_n_p(v) * (1 - n) - beta_n_p(v) * n
        j11 = -alpha_n(v) - beta_n(v)
        return np.array([[j00, j01], [j10, j11]]) / c

    points = {'g': [], 'r': [], 'k': [], 'm': [], 'b': []}
    for i_ext in i_ext_vec:
        for v in find_fixed_points(i_ext):
            n = n_inf(v)
            e = np.linalg.eigvals(jacobian(v, n))
            if abs(e[0].imag) > 1e-6:
                if e[0].real > 0:
                    points['g'].append((i_ext, v))
                if e[0].real < 0:
                    points['r'].append((i_ext, v))
            if abs(e[0].imag) < 1e-6:
                if e[0].real < 0 and e[1].real < 0:
                    points['k'].append((i_ext, v))
                if e[0].real * e[1].real <= 0:
                    points['m'].append((i_ext, v))
                if e[0].real > 0 and e[1].real > 0:
                    points['b'].append((i_ext, v))
    return points, i_ext_vec


def plot_erisir_2d_fp(points, i_ext_vec):
    plt.figure(figsize=(7, 7))
    for color, pts in points.items():
        if not pts:
            continue
        i_pts, v_pts = zip(*pts)
        if color in ('m', 'b'):
            plt.plot(i_pts, v_pts, '--' + color, linewidth=2)
        else:
            plt.plot(i_pts, v_pts, '.' + color, markersize=3)
    plt.xlim(min(i_ext_vec), max(i_ext_vec))
    plt.ylim(-90, 0)
    plt.xlabel(r'$I$ [$\mu$A/cm$^2$]')
    plt.ylabel(r'$v_\ast$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_erisir_2d_fp(*simulate_erisir_2d_fp())

## Erisir Reduced Model: 3D vs 2D

In [ ]:
def simulate_erisir_reduced(c=1.0, g_k=224.0, g_na=112.0, g_l=0.5,
                             v_k=-90.0, v_na=60.0, v_l=-70.0,
                             i_ext=7.0, t_final=100.0, dt=0.01):
    def alpha_h(v):
        return 0.0035 / exp(v / 24.186)

    def alpha_m(v):
        return 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)

    def alpha_n(v):
        return (95 - v) / (exp((95 - v) / 11.8) - 1)

    def beta_h(v):
        return -0.017 * (v + 51.25) / (exp(-(v + 51.25) / 5.2) - 1)

    def beta_m(v):
        return 1.2262 / exp(v / 42.248)

    def beta_n(v):
        return 0.025 / exp(v / 22.222)

    def h_inf(v):
        return alpha_h(v) / (alpha_h(v) + beta_h(v))

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def derivative_3d(x0, t):
        """three-dimensional model: v, h, n dynamic, m=m_inf(v) quasi-static"""
        v, n, h = x0
        m = m_inf(v)
        dv = (g_k * n ** 2 * (v_k - v) + g_na * m ** 3 * h * (v_na - v)
              + g_l * (v_l - v) + i_ext) / c
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return [dv, dn, dh]

    h_plus_n = 0.36

    def derivative_2d(x0, t):
        """reduced (v, n) model: m=m_inf(v), h=h_plus_n-n"""
        v, n = x0
        m = m_inf(v)
        h = h_plus_n - n
        dv = (g_k * n ** 2 * (v_k - v) + g_na * m ** 3 * h * (v_na - v)
              + g_l * (v_l - v) + i_ext) / c
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        return [dv, dn]

    v0 = -70.0
    x0_3d = [v0, n_inf(v0), h_inf(v0)]
    x0_2d = [v0, n_inf(v0)]
    t = np.arange(0, t_final, dt)
    v_3d = odeint(derivative_3d, x0_3d, t)[:, 0]
    v_2d = odeint(derivative_2d, x0_2d, t)[:, 0]
    return t, v_3d, v_2d


def plot_erisir_reduced(t, v_3d, v_2d, t_final=100.0):
    fig, ax = plt.subplots(2, figsize=(8, 7), sharex=True)
    ax[0].plot(t, v_3d, lw=2, c='k')
    ax[0].set_ylabel('$v$ [mV]')
    ax[0].set_title('three-dimensional model')
    ax[1].plot(t, v_2d, lw=2, c='k')
    ax[1].set_xlabel('$t$ [ms]')
    ax[1].set_ylabel('$v$ [mV]')
    ax[1].set_title('two-dimensional model ($h=0.36-n$)')
    for a in ax:
        a.set_xlim(0, t_final)
        a.set_ylim(-95, 55)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_erisir_reduced(*simulate_erisir_reduced())

## Reduced HH: Fixed-Point Count vs $I_{ext}$

In [ ]:
def simulate_hh_reduced_count_fp(g_na=120.0, g_k=36.0, g_l=0.3, v_na=45.0, v_k=-82.0, v_l=-59.0, i_ext_vec=None):
    if i_ext_vec is None:
        i_ext_vec = np.arange(1001) / 1000 * 15
    v = -100 + np.arange(10001) / 10000 * 150
    num_fp = np.zeros(len(i_ext_vec), dtype=int)
    for ijk, i_ext in enumerate(i_ext_vec):
        f = (g_na * hh_reduced_m_inf(v) ** 3 * (0.83 - hh_reduced_n_inf(v)) * (v_na - v)
             + g_k * hh_reduced_n_inf(v) ** 4 * (v_k - v) + g_l * (v_l - v) + i_ext)
        num_fp[ijk] = np.sum(f[:-1] * f[1:] <= 0)
    return num_fp, i_ext_vec

In [ ]:
num_fp, _i_ext_vec = simulate_hh_reduced_count_fp()
print("max fixed points:", num_fp.max(), " min fixed points:", num_fp.min())

## Reduced HH: Attracting/Repelling Cycle Gap vs $I_{ext}$

Three panels at $I=5.5,5.4,5.3$ zoom in on the same fixed point, showing
the attracting (settled) trajectory and the repelling cycle traced
backward from just outside it.

In [ ]:
def simulate_hh_reduced_cycle_distance(i_ext_vec=(5.5, 5.4, 5.3), t_final=1000.0, dt=0.01):
    panels = []
    for i_ext in i_ext_vec:
        v_c = hh_reduced_find_fixed_point(i_ext)
        n_c = hh_reduced_n_inf(v_c)

        v_attr, n_attr = hh_reduced_simulate(20.0, n_c, i_ext, t_final, dt, direction=1)
        tail = round((t_final - 20) / dt)
        v_attr, n_attr = v_attr[tail:], n_attr[tail:]

        v_rep, n_rep = hh_reduced_simulate(v_c + 0.001, n_c, i_ext, t_final, dt, direction=-1)
        tail = round((t_final - 15) / dt)
        v_rep, n_rep = v_rep[tail:], n_rep[tail:]

        panels.append((i_ext, v_c, n_c, v_attr, n_attr, v_rep, n_rep))
    return panels


def plot_hh_reduced_cycle_distance(panels):
    fig, ax = plt.subplots(1, 3, figsize=(12, 4.5))
    for a, (i_ext, v_c, n_c, v_attr, n_attr, v_rep, n_rep) in zip(ax, panels):
        a.plot(v_attr, n_attr, '-k', linewidth=2)
        a.plot(v_c, n_c, '.', markersize=25)
        a.plot(v_rep, n_rep, ':r', linewidth=2)
        a.set_xlim(-67.5, -65.5)
        a.set_ylim(0.3575, 0.3675)
        a.set_xticks([-67, -66])
        a.set_yticks([0.36, 0.365])
        a.set_xlabel('$v$')
        a.set_title(f'$I={i_ext}$')
        a.set_box_aspect(1)
    ax[0].set_ylabel('$n$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_reduced_cycle_distance(simulate_hh_reduced_cycle_distance())

## Reduced HH: Fixed Points and Their Stability

In [ ]:
def simulate_hh_reduced_fixed_points(i_ext_vec=None):
    if i_ext_vec is None:
        i_ext_vec = np.arange(1001) / 1000 * 15
    red, green = [], []
    for i_ext in i_ext_vec:
        v_c = hh_reduced_find_fixed_point(i_ext)
        n_c = hh_reduced_n_inf(v_c)
        e = np.linalg.eigvals(hh_reduced_jacobian(v_c, n_c))
        if e[0].real < 0 and e[1].real < 0 and abs(e[0].imag) > 1e-4:
            red.append((i_ext, v_c))
        if e[0].real > 0 and e[1].real > 0 and abs(e[0].imag) > 1e-4:
            green.append((i_ext, v_c))
    return red, green, i_ext_vec


def plot_hh_reduced_fixed_points(red, green, i_ext_vec):
    plt.figure(figsize=(8, 5))
    if red:
        i_pts, v_pts = zip(*red)
        plt.plot(i_pts, v_pts, '.r', markersize=6)
    if green:
        i_pts, v_pts = zip(*green)
        plt.plot(i_pts, v_pts, '--g', linewidth=3)
    plt.xlabel('$I$')
    plt.ylabel(r'$v_\ast$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_reduced_fixed_points(*simulate_hh_reduced_fixed_points())

## Reduced HH: Fixed-Point Eigenvalues

In [ ]:
def simulate_hh_reduced_fp_evs(i_ext_vec=None):
    if i_ext_vec is None:
        i_ext_vec = np.arange(1001) / 1000 * 15
    real_part = np.zeros(len(i_ext_vec))
    imag_part = np.zeros(len(i_ext_vec))
    for ijk, i_ext in enumerate(i_ext_vec):
        v_c = hh_reduced_find_fixed_point(i_ext)
        n_c = hh_reduced_n_inf(v_c)
        e = np.linalg.eigvals(hh_reduced_jacobian(v_c, n_c))
        real_part[ijk] = e[0].real
        imag_part[ijk] = e[0].imag
    return real_part, imag_part, i_ext_vec


def plot_hh_reduced_fp_evs(real_part, imag_part):
    A, B, C, D = -1, 1, -1, 1
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(real_part, imag_part, color='k', linewidth=2)
    ax.plot(real_part, -imag_part, color='k', linewidth=2)

    ind = np.where((real_part[:-1] < 0.1) & (real_part[1:] > 0.1))[0][0]
    x, y = real_part[ind], imag_part[ind]
    v = np.array([real_part[ind] - real_part[ind - 200], imag_part[ind] - imag_part[ind - 200]])
    draw_arrow(ax, [A, B], [C, D], x, y, v, epsilon=0.1, width=2)
    draw_arrow(ax, [A, B], [C, D], x, -y, [v[0], -v[1]], epsilon=0.1, width=2)

    ax.plot([0, 0], [-1, 1], color='b', linewidth=1)
    ax.set_xlim(A, B)
    ax.set_ylim(C, D)
    ax.set_box_aspect(1)
    ax.set_xlabel(r'${\rm Re}\,(\lambda)$')
    ax.set_ylabel(r'${\rm Im}\,(\lambda)$')
    plt.tight_layout()
    plt.show()

In [ ]:
real_part, imag_part, _i_ext_vec = simulate_hh_reduced_fp_evs()
plot_hh_reduced_fp_evs(real_part, imag_part)

## Reduced HH: Repelling Cycle Around the Fixed Point

In [ ]:
def simulate_hh_reduced_repelling_cycle(i_ext=5.5, dt=0.01):
    v_c = hh_reduced_find_fixed_point(i_ext)
    n_c = hh_reduced_n_inf(v_c)

    v_attr, n_attr = hh_reduced_simulate(20.0, n_c, i_ext, 1000.0, dt, direction=1)
    tail = round(20 / dt)
    v_attr, n_attr = v_attr[-tail:], n_attr[-tail:]

    v_rep, n_rep = hh_reduced_simulate(v_c + 0.001, n_c, i_ext, 1000.0, dt, direction=-1)
    tail = round(15 / dt)
    v_rep, n_rep = v_rep[-tail:], n_rep[-tail:]

    return v_c, n_c, v_attr, n_attr, v_rep, n_rep, dt


def plot_hh_reduced_repelling_cycle(v_c, n_c, v_attr, n_attr, v_rep, n_rep):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].plot(v_attr, n_attr, color='k', linewidth=2)
    axes[0].plot(v_rep, n_rep, ':r', linewidth=2)
    axes[0].set_xlim(-100, 50)
    axes[0].set_ylim(0.3, 0.8)
    axes[0].set_box_aspect(1)
    axes[0].set_xlabel('$v$')
    axes[0].set_ylabel('$n$')

    axes[1].plot(v_attr, n_attr, color='k', linewidth=2)
    axes[1].plot(v_c, n_c, '.', markersize=15)
    axes[1].plot(v_rep, n_rep, ':r', linewidth=2)
    axes[1].set_xlim(-75, -55)
    axes[1].set_ylim(0.35, 0.45)
    axes[1].set_box_aspect(1)
    axes[1].set_xlabel('$v$')
    axes[1].set_ylabel('$n$')

    plt.tight_layout()
    plt.show()

In [ ]:
v_c, n_c, v_attr, n_attr, v_rep, n_rep, _dt = simulate_hh_reduced_repelling_cycle()
plot_hh_reduced_repelling_cycle(v_c, n_c, v_attr, n_attr, v_rep, n_rep)